# Self-Reflection & Critique: Engineering Autonomous LLM Reflection
### Practical implementations of every concept from the BIA deck

**How LLMs diagnose, refine, and improve outputs before agents scale up.**

This notebook turns each slide of the PDF into runnable code:

| # | Section | Slide concept |
|---|---------|---------------|
| 1 | Setup & helper functions | — |
| 2 | The motivating problem | *The Brief vs The First Draft* |
| 3 | Self-Reflection vs Critique | *The useful unit is "what should change, and why?"* |
| 4 | The 5 ingredients of a useful critique | *Quality Check pentagon* |
| 5 | Weak vs Strong critique prompts | *Critique quality depends on review criteria* |
| 6 | **Pattern 1: Self-Refine** | Generator → Feedback → Refiner loop |
| 7 | **Pattern 2: Reflexion** | Language feedback as memory, not gradients |
| 8 | Reflection Memory Triage | Store vs Avoid + Refresh protocol |
| 9 | **Pattern 3: Evaluator-Generator** | Threshold gate + structured scores |
| 10 | The Evaluator Engine | Rubric → JSON (`score`, `passed`, `issues`, `revision_instructions`) |
| 11 | **Pattern 4: Principles-Based Critique** | The Constitution + Critic Agent |
| 12 | Stopping Rules (Circuit Breakers) | Pass threshold, max iterations, no-improvement, escalation |
| 13 | **Capstone: The Writing Critic Agent** | Rubric + Constitution + Refiner + Trace |
| 14 | The Blueprint Matrix + exercises | Choosing your engine |

> **Core Directive (from the deck):** *Make quality improvement systematic, not accidental.*


### 1. Setup & helper functions 

In [3]:
import os, json, re, textwrap
from dataclasses import dataclass, field
from typing import Optional
from litellm import completion
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
MODEL = "gpt-4o-mini"  

In [6]:
def llm(prompt: str, system: str = "You are a helpful assistant.",
        temperature: float = 0.7, json_mode: bool = False) -> str:
    """Single LLM call. json_mode=True nudges + parses strict JSON output."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": prompt}]
    resp = completion(model=MODEL, messages=messages, temperature=temperature)
    return resp.choices[0].message.content.strip()


In [7]:
def llm_json(prompt: str, system: str, temperature: float = 0.0) -> dict:
    """LLM call that must return JSON. Strips markdown fences and parses."""
    raw = llm(prompt, system=system + "\nRespond ONLY with valid JSON. No preamble, no markdown fences.",
              temperature=temperature)
    cleaned = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        # one repair attempt — a common production trick
        repaired = llm(f"Fix this into strict valid JSON, output only the JSON:\n{raw}",
                       system="You repair malformed JSON.", temperature=0.0)
        repaired = re.sub(r"^```(?:json)?|```$", "", repaired.strip(), flags=re.MULTILINE).strip()
        return json.loads(repaired)


def show(text, width=100):
    print(textwrap.fill(text, width=width, replace_whitespace=False))

In [9]:
show(llm("Hi", "You are helpful assitance to answer any query"))

Hello! How can I assist you today?


In [10]:
show(llm("Hi", "You are helpful assitance to answer any query"))

Hello! How can I assist you today?


### 2 The motivating problem


*"Is there an answer?" vs "Does this answer satisfy the brief?"*

**The Brief:** *Write a short email inviting working professionals to a weekend GenAI workshop. Tone: credible, specific, not hype-heavy.*

A single-shot LLM call reliably produces **an** answer. The deck's annotated first draft shows exactly why that's not enough:


| First draft fragment | Diagnosed defect |
|---|---|
| "Join our amazing AI workshop!" | Audience is too broad |
| "Learn everything about GenAI" | No concrete outcome |
| "...become an expert" | Overpromises expertise |
| "...transform your career." | No date / format / CTA |


> **Reflection starts when we stop asking "is there an answer?" and start asking "does this answer satisfy the actual brief?"**


Let's reproduce the failure first.



In [11]:
BRIEF = "Write a short email inviting working professionals to a weekend GenAI workshop"

first_draft = llm(f"Write an email for this: {BRIEF}", temperature=1.0)
show(first_draft)


Subject: Join Us for a Weekend GenAI Workshop!

Dear [Recipient's Name],

I hope this message finds
you well! I am excited to invite you to an engaging workshop on Generative AI taking place this
weekend. 

Whether you're looking to enhance your skills, explore innovative applications, or
network with fellow professionals in the field, this workshop offers a great opportunity to dive
into the fascinating world of GenAI.

**Workshop Details:**
- **Date:** [Insert Date]
- **Time:**
[Insert Time]
- **Location:** [Insert Location/Link for Virtual]
- **Registration Fee:** [Insert Fee
if applicable]

Don’t miss out on this chance to learn from industry experts and collaborate with
peers. Please RSVP by [Insert RSVP Deadline] to confirm your attendance.

Looking forward to seeing
you there!

Best regards,

[Your Name]  
[Your Position]  
[Your Company]  
[Your Contact
Information]


Run the cell a few times. You'll typically see the exact defect classes the deck highlights — hype adjectives, vague outcomes, missing logistics. The rest of this notebook is about **catching and fixing those defects systematically**.

### Where this fits in the reasoning arc

```
1. Prompt (clear task) → 2. Reason (structured path) → 3. Act (tool/output)
        ↑                                                      │
        └──────── 5. Revise (improve) ← 4. Critique (detect gaps)
```

Steps 4 and 5 are the subject of this notebook. The dashed arrows in the deck matter: critique can route back to the **prompt** (re-specify the task), not only to the draft.


---
## 3. Self-Reflection vs Critique — precise definitions

The deck draws a jigsaw between two interlocking concepts:

- **Self-reflection** — *an LLM-generated review of an output against the task goal, constraints, and quality criteria.* (The act of looking.)
- **Critique** — *a diagnosis that names specific defects and gives revision instructions.* (The actionable artifact.)

> **The useful unit is not "confidence." The useful unit is "what should change, and why?"**

A reflection that says *"This looks good, 8/10, I'm fairly confident"* is useless to a refiner. A critique that says *"The subject line doesn't state the date; move '25 July' into it"* is directly executable. Let's demonstrate the difference on the same draft.


In [12]:
# A confidence-style reflection (what we DON'T want)
confidence_reflection = llm(
    f"Review this email and say how confident you are that it is good:\n\n{first_draft}",
    temperature=0.3)

print("=== CONFIDENCE-STYLE REFLECTION (weak) ===")
show(confidence_reflection)

=== CONFIDENCE-STYLE REFLECTION (weak) ===
I would rate this email as quite good, with a confidence level of about 85%. Here are some strengths
and areas for improvement:

### Strengths:
1. **Clear Subject Line**: The subject line is engaging
and clearly states the purpose of the email.
2. **Friendly Tone**: The opening line is warm and
inviting, which sets a positive tone for the invitation.
3. **Purpose and Benefits**: The email
clearly outlines the purpose of the workshop and the benefits of attending, which can motivate
recipients to participate.
4. **Structured Information**: The workshop details are presented in a
clear and organized manner, making it easy for the reader to find essential information.
5. **Call
to Action**: The request for an RSVP is clear, encouraging recipients to respond.

### Areas for
Improvement:
1. **Personalization**: While there is a placeholder for the recipient's name, ensuring
that it is personalized will enhance engagement.
2. **Specific Details**: T

In [13]:
change_critique = llm(
    f"""Here is a brief and a draft.

BRIEF:
{BRIEF}

DRAFT:
{first_draft}

List the specific defects (what is missing, incorrect, vague, or risky relative to the brief),
and for each defect give a concrete revision instruction a writer could apply mechanically.""",
    temperature=0.3)

print("=== CHANGE-ORIENTED CRITIQUE (strong) ===")
show(change_critique)

=== CHANGE-ORIENTED CRITIQUE (strong) ===
Here are the specific defects identified in the draft, along with concrete revision instructions:
1. **Missing Personalization**: The draft uses a placeholder "[Recipient's Name]" but does not
provide a method for personalization.
   - **Revision Instruction**: Replace "[Recipient's Name]"
with a dynamic field that automatically inserts the recipient's name when sending the email.

2.
**Vague Workshop Details**: The placeholders for date, time, location, and registration fee are not
filled in, making it unclear for the recipient.
   - **Revision Instruction**: Fill in the specific
details for the workshop, including the exact date, time, location (or link for virtual attendance),
and any applicable registration fee.

3. **Lack of Clear Call to Action**: The phrase "Please RSVP
by [Insert RSVP Deadline]" is vague and does not specify how to RSVP.
   - **Revision Instruction**:
Specify the RSVP deadline and provide clear instructions on how to RS

---
## 4. A Useful Critique Has Five Ingredients

The deck's pentagon, as a data structure. Every quality check should carry:

1. **Task goal** — what the output is supposed to achieve
2. **Constraints** — format, length, audience, tone, safety limits
3. **Rubric** — *observable* criteria for quality
4. **Specific defects** — what is missing, incorrect, vague, or risky
5. **Revision instructions** — concrete changes the refiner can apply

Ingredients 1–3 are the **inputs** to a quality check; 4–5 are its **outputs**. Encoding this as dataclasses makes the contract explicit and reusable across all four patterns.


In [14]:
class Product:
    # name: str
    # price: float
    # quantity: int = 0  # Default value

    def __init__(self, name, price, product):
        self.name= name
        self.price = price
        self.product = product


    def show(self):
        print(self.name)
        print(self.price)
        print(self.product)



In [15]:
obj = Product("rajesh", 1.2, "genai")

In [16]:
obj.show()

rajesh
1.2
genai


In [17]:
feedback = {
  "feedback": [
    {
      "issue": "Missing Personalization",
      "defect": "The draft uses a generic greeting (\"Dear [Recipient's Name]\") without any personalization or context.",
      "revision_instruction": "Replace \"[Recipient's Name]\" with a specific name or use a more personalized greeting, such as \"Dear Colleagues\" or \"Dear [Team/Department Name]\"."
    },
    {
      "issue": "Incomplete Date and Time Information",
      "defect": "The placeholders for date and time (\"[Insert Date]\" and \"[Insert Time]\") are not filled in.",
      "revision_instruction": "Insert the specific date and time of the workshop in place of the placeholders."
    },
    {
      "issue": "Vague Location Information",
      "defect": "The location is indicated as a placeholder (\"[Insert Location/Online Platform]\") without specifics.",
      "revision_instruction": "Provide the exact location or specify the online platform (e.g., Zoom, Microsoft Teams) where the workshop will be held."
    },
    {
      "issue": "Lack of RSVP Details",
      "defect": "The RSVP deadline is a placeholder (\"[Insert RSVP Deadline]\") and lacks specifics.",
      "revision_instruction": "Insert a specific date and time for the RSVP deadline to encourage timely responses."
    },
    {
      "issue": "Missing Call to Action",
      "defect": "While there is an RSVP request, it could be more compelling and direct.",
      "revision_instruction": "Strengthen the call to action by adding a phrase like \"Don't miss out on this opportunity—reserve your spot today!\" before the RSVP line."
    },
    {
      "issue": "No Mention of Benefits or Outcomes",
      "defect": "The benefits of attending the workshop are somewhat vague and could be more compelling.",
      "revision_instruction": "Add specific outcomes or skills participants will gain, such as \"You will learn how to implement AI tools in your projects, improve efficiency, and drive innovation in your workplace.\""
    },
    {
      "issue": "Missing Contact Information",
      "defect": "The contact information is indicated as a placeholder (\"[Your Contact Information]\") and may not include all necessary details.",
      "revision_instruction": "Ensure that your full contact information is included, such as your phone number and email address, to facilitate communication."
    },
    {
      "issue": "Subject Line Could Be More Engaging",
      "defect": "The subject line is straightforward but could be more engaging to attract attention.",
      "revision_instruction": "Revise the subject line to something like \"Unlock the Future: Join Our Weekend GenAI Workshop!\" to create excitement."
    }
  ]
}

In [18]:
feedback['feedback'][0]['defect']

'The draft uses a generic greeting ("Dear [Recipient\'s Name]") without any personalization or context.'

In [19]:
@dataclass
class TaskSpec:
    """Ingredients 1-3: what the critic needs BEFORE it can judge anything."""
    goal: str                          # 1. Task goal
    constraints: list[str]             # 2. Constraints
    rubric: dict[str, str]             # 3. Rubric: criterion -> observable definition

    def render(self) -> str:
        lines = [f"TASK GOAL: {self.goal}", "CONSTRAINTS:"]
        lines += [f"  - {c}" for c in self.constraints]
        lines.append("RUBRIC (observable criteria):")
        lines += [f"  - {k}: {v}" for k, v in self.rubric.items()]
        return "\n".join(lines)



In [20]:
@dataclass
class Critique:
    """Ingredients 4-5: what the critic must produce."""
    score: int                         # 0-10, enables thresholds
    passed: bool                       # routes the loop
    defects: list[str]                 # 4. Specific defects (traceable debugging evidence)
    revision_instructions: str         # 5. Concrete changes the refiner can apply


In [21]:
EMAIL_SPEC = TaskSpec(
    goal="Invite working professionals to a weekend GenAI workshop and get them to register.",
    constraints=[
        "Short: under 160 words",
        "Audience: mid-career tech professionals (data/engineering) in India",
        "Tone: credible, specific, not hype-heavy — no words like 'amazing', 'revolutionary'",
        "Must include: date (Sat 25 July 2026), format (in-person, Bengaluru, 10am-5pm)",
        "Must include: exactly one concrete learning outcome",
        "Must include: a clear CTA with a link placeholder [REGISTER_LINK]",
    ],
    rubric={
        "clarity":       "A reader knows what/when/where within 5 seconds",
        "specificity":   "Names a concrete skill/outcome, not 'learn everything'",
        "factual_risk":  "No overpromises ('become an expert', guaranteed jobs)",
        "audience_fit":  "References the reader's actual work context",
        "cta":           "One unambiguous next action with the link placeholder",
    },
)

print(EMAIL_SPEC.render())



TASK GOAL: Invite working professionals to a weekend GenAI workshop and get them to register.
CONSTRAINTS:
  - Short: under 160 words
  - Audience: mid-career tech professionals (data/engineering) in India
  - Tone: credible, specific, not hype-heavy — no words like 'amazing', 'revolutionary'
  - Must include: date (Sat 25 July 2026), format (in-person, Bengaluru, 10am-5pm)
  - Must include: exactly one concrete learning outcome
  - Must include: a clear CTA with a link placeholder [REGISTER_LINK]
RUBRIC (observable criteria):
  - clarity: A reader knows what/when/where within 5 seconds
  - specificity: Names a concrete skill/outcome, not 'learn everything'
  - factual_risk: No overpromises ('become an expert', guaranteed jobs)
  - audience_fit: References the reader's actual work context
  - cta: One unambiguous next action with the link placeholder


In [22]:
EMAIL_SPEC.constraints

['Short: under 160 words',
 'Audience: mid-career tech professionals (data/engineering) in India',
 "Tone: credible, specific, not hype-heavy — no words like 'amazing', 'revolutionary'",
 'Must include: date (Sat 25 July 2026), format (in-person, Bengaluru, 10am-5pm)',
 'Must include: exactly one concrete learning outcome',
 'Must include: a clear CTA with a link placeholder [REGISTER_LINK]']

---
## 5. Weak vs Strong Critique Prompts

The deck's side-by-side:

**Weak:** `"Check this and improve it."`
- ❌ no criteria &nbsp; ❌ no role separation &nbsp; ❌ no required output format &nbsp; ❌ no stopping signal

**Strong:** *"Evaluate the draft against the brief. Score clarity, specificity, factual risk, audience fit, and CTA. Return JSON with score, issues, and revision instructions."*
- ✅ task goal & constraints &nbsp; ✅ rubric &nbsp; ✅ defects & revision instructions (in a machine-parseable format)

> **Critique quality depends more on the review criteria than on the word "critique."**

Run both against the same draft and compare.


In [23]:
# WEAK critique prompt
weak = llm(f"Check this and improve it:\n\n{first_draft}", temperature=0.3)
print("=== WEAK PROMPT OUTPUT ===")
show(weak[:800])

=== WEAK PROMPT OUTPUT ===
Subject: Join Us for an Exciting Weekend Workshop on Generative AI!

Dear [Recipient's Name],

I
hope this message finds you in great spirits! I am thrilled to invite you to a dynamic workshop on
Generative AI happening this weekend.

Whether you're eager to enhance your skills, discover
innovative applications, or connect with fellow professionals in the field, this workshop is the
perfect opportunity to immerse yourself in the captivating world of GenAI.

**Workshop Details:**
-
**Date:** [Insert Date]
- **Time:** [Insert Time]
- **Location:** [Insert Location/Link for Virtual]
- **Registration Fee:** [Insert Fee if applicable]

Don’t miss this chance to learn from industry
experts and collaborate with like-minded peers. Please RSVP by [Insert RSVP Deadline] to secure your
spot.

I look


In [24]:
# STRONG critique prompt — criteria + role separation + output format + stopping signal
CRITIC_SYSTEM = "You are a strict writing critic. You never rewrite the draft; you only diagnose and instruct."

def strong_critique(draft: str, spec: TaskSpec, threshold: int = 8) -> Critique:
    prompt = f"""{spec.render()}

DRAFT:
\"\"\"{draft}\"\"\"

Evaluate the draft against the task goal, constraints, and rubric above.
Return JSON with exactly these keys:
  "score": integer 0-10 (10 = fully satisfies brief),
  "passed": boolean (true only if score >= {threshold} AND all 'Must include' constraints are met),
  "defects": array of short strings, each naming ONE specific defect,
  "revision_instructions": a single string of concrete, mechanical edits."""
    data = llm_json(prompt, system=CRITIC_SYSTEM)
    return Critique(**{k: data[k] for k in ("score", "passed", "defects", "revision_instructions")})

crit = strong_critique(first_draft, EMAIL_SPEC)

print("=== STRONG PROMPT OUTPUT (parsed Critique object) ===")
print(f"score  = {crit.score}   passed = {crit.passed}")
print("defects:")
for d in crit.defects: print("  •", d)
print("\nrevision_instructions:")
show(crit.revision_instructions)


=== STRONG PROMPT OUTPUT (parsed Critique object) ===
score  = 3   passed = False
defects:
  • Date, time, and location are not specified.
  • No concrete learning outcome mentioned.
  • Tone is too casual and includes hype language.
  • No clear call to action with a link placeholder.

revision_instructions:
Replace placeholders with specific details: Date (Sat 25 July 2026), Time (10am-5pm), Location
(Bengaluru). Include a concrete learning outcome related to Generative AI. Use a more formal tone
and remove hype language. Add a clear call to action with [REGISTER_LINK].


Compare the four errors of the weak prompt against what we just fixed:

| Weak-prompt error | How the strong prompt fixes it |
|---|---|
| No criteria | `spec.render()` injects goal + constraints + rubric |
| No role separation | Critic system prompt: *"You never rewrite the draft"* |
| No required output format | Strict JSON schema, parsed into a `Critique` dataclass |
| No stopping signal | `passed` boolean tied to an explicit threshold |

The weak prompt also **conflates critic and refiner** — it rewrites the draft, so you get neither a reusable diagnosis nor a controlled revision. Role separation is the seed of Pattern 3.


---
## 6. Pattern 1: Self-Refine

Three roles, one loop:

```
        ┌──────────────────────────────┐
        │  GENERATOR: creates initial  │
        │  answer                      │
        └──────────────┬───────────────┘
                       ▼
        ┌──────────────────────────────┐
        │  FEEDBACK PROVIDER: finds    │
        │  issues and suggests fixes   │
        └──────────────┬───────────────┘
                       ▼
        ┌──────────────────────────────┐
        │  REFINER: produces improved  │──▶ loop back to feedback
        │  answer                      │    until stop condition
        └──────────────────────────────┘
```

All three roles can be **the same model with different prompts** — that's the "self" in Self-Refine (Madaan et al., 2023).

**Fit guidance from the deck:**

| ✅ Good fit (checklist-style) | ⚠️ Weak fit alone |
|---|---|
| Drafting, editing, summaries, format repair, checklist-style reviews | Unknown factual truth, hidden math errors, security-sensitive outputs |

> **Rule of thumb: Reflection improves alignment to *stated criteria*; it does not create *missing evidence*.**

If the model doesn't know a fact, reflecting harder won't conjure it — it may just make the hallucination more confident. (That's why the stopping rules in Section 12 include *escalate to human review*.)


In [25]:
def self_refine(brief: str, spec: TaskSpec, max_iterations: int = 3,
                threshold: int = 8, verbose: bool = True):
    """Pattern 1: Generator -> Feedback -> Refiner loop with a stop condition."""
    trace = []

    # --- GENERATOR ---
    draft = llm(f"{spec.render()}\n\nBRIEF:\n{brief}\n\nWrite the email.",
                system="You are a marketing copywriter.", temperature=0.8)

    for i in range(1, max_iterations + 1):
        # --- FEEDBACK PROVIDER ---
        crit = strong_critique(draft, spec, threshold=threshold)
        trace.append({"iteration": i, "score": crit.score,
                      "n_defects": len(crit.defects), "defects": crit.defects})
        if verbose:
            print(f"[iter {i}] score={crit.score} passed={crit.passed} "
                  f"defects={len(crit.defects)}")

        if crit.passed:                                   # stopping signal
            break

        # --- REFINER ---
        draft = llm(
            f"""{spec.render()}

CURRENT DRAFT:
\"\"\"{draft}\"\"\"

CRITIQUE — apply these revision instructions exactly, change nothing else:
{crit.revision_instructions}

Output only the revised email.""",
            system="You are a careful editor. You apply instructions; you do not re-invent.",
            temperature=0.4)

    return draft, trace


final_email, trace = self_refine(BRIEF, EMAIL_SPEC)
print("\n=== FINAL EMAIL ===")
show(final_email)

[iter 1] score=8 passed=True defects=0

=== FINAL EMAIL ===
Subject: Join Us for a Weekend GenAI Workshop in Bengaluru

Dear [Recipient's Name],

We invite you
to enhance your skill set and explore the practical applications of Generative AI at our workshop on
Saturday, 25 July 2026, from 10 AM to 5 PM in Bengaluru.

This in-person event is designed
specifically for mid-career tech professionals like you. You will learn how to implement GenAI
models to optimize data processes in your projects, a skill that can be directly applied in your
work.

Don’t miss this opportunity to stay ahead in your field and network with peers.
[REGISTER_LINK]

Best regards,  
[Your Name]  
[Your Company]  
[Your Contact Information]


Pattern 2: Reflexion — *language feedback, not gradient updates*

```
ORDINARY RETRY:   attempt → fail → ask again          ──▶ (repeats the same mistake)

REFLEXION LOOP:   attempt → feedback → MEMORY NODE ───▶ next attempt
                                       (reflection)        (informed by lesson)
```

In [26]:
# A task with an EXTERNAL verifier: unit tests. (Ground-truth feedback, not LLM opinion.)
CODING_TASK = """Write a Python function `parse_price(s: str) -> float` that parses Indian-format
price strings into floats. Examples it must handle:
  "₹1,23,456.50" -> 123456.50      (Indian digit grouping)
  "Rs. 999"      -> 999.0
  " 1,000 "      -> 1000.0
  "₹0"           -> 0.0
Return only the function code, no explanation."""


TESTS = [("₹1,23,456.50", 123456.50), ("Rs. 999", 999.0), (" 1,000 ", 1000.0), ("₹0", 0.0)]




def run_tests(code_str: str):
    """External verifier. Returns (all_passed, feedback_string)."""
    ns = {}
    try:
        cleaned = re.sub(r"^```(?:python)?|```$", "", code_str.strip(), flags=re.MULTILINE)
        print(cleaned)
        exec(cleaned, ns)
        fn = ns["parse_price"]
    except Exception as e:
        return False, f"Code failed to load: {type(e).__name__}: {e}"
    failures = []
    for inp, expected in TESTS:
        try:
            got = fn(inp)
            if abs(got - expected) > 1e-6:
                failures.append(f"parse_price({inp!r}) returned {got!r}, expected {expected!r}")
        except Exception as e:
            failures.append(f"parse_price({inp!r}) raised {type(e).__name__}: {e}")
    return (len(failures) == 0), ("All tests passed." if not failures else "\n".join(failures))


In [28]:
run_tests(python_func)

NameError: name 'python_func' is not defined

In [ ]:
exec(python_func)

parse_price("Rs. 999")

999.0

: 

: 

In [29]:
python_func = """
def parse_price(s: str) -> float:
    s = s.strip()
    s = s.replace('₹', '')
    s = s.replace('Rs.', '')
    s = s.replace('Rs', '')
    s = s.replace(',', '')
    return float(s)
"""

exec(python_func)

print(parse_price("₹1,23,456.50"))


123456.5


In [30]:
def ordinary_retry(task: str, max_attempts: int = 3):
    """Baseline: attempt -> fail -> ask again (no memory). Often repeats the same mistake."""
    for i in range(1, max_attempts + 1):
        code_out = llm(task, system="You are a Python developer.", temperature=0.8)
        print(code_out)
        ok, fb = run_tests(code_out)
        print(f"[ordinary retry {i}] passed={ok}")
        if ok: return code_out, i
    return code_out, max_attempts

In [31]:
ordinary_retry("Write a Python function `parse_price(s: str) -> float` that parses Indian-format")

To parse prices in Indian format (which typically uses comma separators for thousands and lakhs, and a decimal point for cents), we can create a Python function `parse_price(s: str) -> float`. 

In the Indian numbering system, the first comma comes after three digits from the right, and subsequent commas come after every two digits. For example, "1,00,000.50" represents one lakh and fifty paise.

Here’s how you could implement such a function:

```python
import re

def parse_price(s: str) -> float:
    # Remove any spaces
    s = s.replace(" ", "")
    
    # Use regex to remove commas
    s = s.replace(',', '')
    
    # Convert the cleaned string to a float
    try:
        price = float(s)
    except ValueError:
        raise ValueError(f"Invalid price format: {s}")

    return price

# Example usage:
print(parse_price("1,23,456.78"))  # 123456.78
print(parse_price("12,34,567"))     # 1234567.0
print(parse_price("1,00,000.00"))   # 100000.0
```

### Explanation:
1. **Removing Space

('To parse an Indian-format price string and return it as a float in Python, we need to account for the common formatting styles used in India. This includes the use of commas as thousand separators, optional currency symbols (like "₹"), and possibly the presence of decimal points for paise (the smallest currency unit in India).\n\nHere\'s a function `parse_price(s: str) -> float` that accomplishes this:\n\n```python\nimport re\n\ndef parse_price(s: str) -> float:\n    # Remove currency symbols and whitespace\n    s = s.replace(\'₹\', \'\').replace(\'Rs.\', \'\').replace(\' \', \'\').strip()\n    \n    # Remove commas\n    s = s.replace(\',\', \'\')\n    \n    # Use regex to ensure we\'re only left with numbers and a decimal point\n    if not re.match(r\'^\\d+(\\.\\d{1,2})?$\', s):\n        raise ValueError("Invalid price format")\n    \n    # Convert to float\n    return float(s)\n\n# Example usage:\nprint(parse_price("₹ 1,23,456.78"))  # Output: 123456.78\nprint(parse_price("Rs. 2,50

In [32]:
def reflexion_loop(task: str, max_attempts: int = 10):
    """Reflexion: attempt -> feedback -> reflect into MEMORY -> next attempt uses lessons."""
    memory: list[str] = []                       # the Memory Node from the deck

    for i in range(1, max_attempts + 1):
        lessons = ("\nLESSONS FROM PREVIOUS FAILED ATTEMPTS:\n" +
                   "\n".join(f"- {m}" for m in memory)) if memory else ""
        code_out = llm(task + lessons, system="You are a Python developer.", temperature=0.8)
        print(f"code-output \n{code_out}")
        ok, fb = run_tests(code_out)             # external verifiable feedback
        print(f"[reflexion {i}] passed={ok}" + (f" | lessons in memory: {len(memory)}"))
        if ok: return code_out, i, memory

        # --- REFLECT: compress the failure into a short lesson (the key mechanic) ---
        lesson = llm(
            f"""Your code attempt failed these tests:
{fb}

The code was:
{code_out}

Write ONE compact lesson (max 30 words) stating WHY it failed and WHAT strategy
the next attempt should use. Be specific and technical, not generic advice.""",
            system="You extract root-cause lessons from failures.", temperature=0.2)
        memory.append(lesson.strip())

    return code_out, max_attempts, memory

In [33]:
solution, attempts, lessons = reflexion_loop(CODING_TASK)

code-output 
```python
import re

def parse_price(s: str) -> float:
    s = s.strip()
    s = re.sub(r'₹|Rs\.?|₹\s*|Rs\s*', '', s)  # Remove currency symbols
    s = re.sub(r',', '', s)  # Remove commas
    return float(s) if s else 0.0
```

import re

def parse_price(s: str) -> float:
    s = s.strip()
    s = re.sub(r'₹|Rs\.?|₹\s*|Rs\s*', '', s)  # Remove currency symbols
    s = re.sub(r',', '', s)  # Remove commas
    return float(s) if s else 0.0

[reflexion 1] passed=True | lessons in memory: 0


In [34]:
solution = """
import re

def parse_price(s: str) -> float:
    # Remove currency symbols and strip whitespace
    s = s.replace('₹', '').replace('Rs.', '').strip()
    # Remove commas using the proper regex for Indian format
    s = re.sub(r'(?<=\d)(?=(\d{2})+(?!\d))', '', s.replace(',', ''))
    # Convert to float and return
    return float(s)
    """

exec(solution)
print(parse_price("9,99.0"))

999.0


In [35]:
print(f"\nSolved in {attempts} attempt(s). Lessons accumulated:")


Solved in 1 attempt(s). Lessons accumulated:


In [36]:
for l in lessons: print("  🧠", l)

In [37]:
exec(solution)

## 9. Pattern 3: Evaluator-Generator — *a quality gate in a pipeline*


```
                     Threshold Gate
                          │
┌───────────────┐  1.draft │   ┌───────────────────┐  4. accepted output
│ GENERATOR NODE│──────────┼──▶│  EVALUATOR NODE   │────────────▶ (if passed)
│ (drafts)      │◀─ ─ ─ ─ ─┼─ ─│  (scores vs rubric)│
└───────────────┘ 3.revision   └───────────────────┘
                  instructions        │ 2. score
                  (if failed)         ▼
```


Differences from Self-Refine:
- **Hard role separation** — evaluator and generator are separate nodes (can even be different models: cheap generator, strong evaluator, or vice-versa).
- **Structured scores** cross the boundary, not free text — which is what lets you put a *gate* in a pipeline (LangGraph conditional edge, Airflow branch, CI check).



In [38]:
EMAIL_SPEC

TaskSpec(goal='Invite working professionals to a weekend GenAI workshop and get them to register.', constraints=['Short: under 160 words', 'Audience: mid-career tech professionals (data/engineering) in India', "Tone: credible, specific, not hype-heavy — no words like 'amazing', 'revolutionary'", 'Must include: date (Sat 25 July 2026), format (in-person, Bengaluru, 10am-5pm)', 'Must include: exactly one concrete learning outcome', 'Must include: a clear CTA with a link placeholder [REGISTER_LINK]'], rubric={'clarity': 'A reader knows what/when/where within 5 seconds', 'specificity': "Names a concrete skill/outcome, not 'learn everything'", 'factual_risk': "No overpromises ('become an expert', guaranteed jobs)", 'audience_fit': "References the reader's actual work context", 'cta': 'One unambiguous next action with the link placeholder'})

In [40]:
BRIEF

'Write a short email inviting working professionals to a weekend GenAI workshop'

In [48]:
from turtle import st


@dataclass
class GateResult:
    accepted: bool
    output: str
    iterations: int
    final_score: int
    trace: list


def evaluator_generator(brief: str, spec: TaskSpec,
                        threshold: int=7, max_iterations: int =10,
                        gen_temperature: float = 0.8) -> GateResult:
    
    """Pattern 3: separate generator and evaluator nodes with a threshold gate."""
    trace = []
    revision_instructions = None

    for i in range(1, max_iterations + 1):
        # --- GENERATOR Node---
        gen_prompt = f"BRIEF:\n{brief}\n\n write the email"

        if revision_instructions:
            gen_prompt += f"\n\nApply these revision instructions to your previous attempt:\n{revision_instructions} so that the new draft addresses the defects and meets the brief."
        draft = llm(gen_prompt, system="you are copywriter.", temperature=gen_temperature)

        #### Evaluator Node ---
        crit = strong_critique(draft, spec, threshold=threshold)

        trace.append({"iteration": i, "score": crit.score,"passed": crit.passed, "n_defects": len(crit.defects), "defects": crit.defects})

        print(f"[gate iter {i}] score={crit.score} passed={crit.passed}")
        if crit.passed:
            return GateResult(accepted=True, output=draft, iterations=i, final_score=crit.score, trace=trace)
        
        revision_instructions = crit.revision_instructions
    print(f"traces {trace}")

    return GateResult(accepted=False, output=draft, iterations=max_iterations, final_score=crit.score, trace=trace)

        



In [49]:
result = evaluator_generator(BRIEF, EMAIL_SPEC)

print(f"\nAccepted: {result.accepted} after {result.iterations} iteration(s), score {result.final_score}")


[gate iter 1] score=4 passed=False
[gate iter 2] score=8 passed=True

Accepted: True after 2 iteration(s), score 8


In [50]:
show(result.output)

Subject: Join Us for a Weekend Workshop on Generative AI

Dear [Recipient's Name],

We are pleased
to invite you to a weekend workshop on Generative AI, designed specifically for working
professionals who want to deepen their understanding of this transformative technology. 

**Date:**
Saturday, 25 July 2026  
**Time:** 10 AM - 5 PM  
**Location:** Bengaluru  

Throughout the
workshop, you will gain practical insights into how Generative AI can enhance your business
processes and creative projects. By the end of the session, you will have the skills to create basic
AI-generated content, enabling you to apply these insights directly in your work.

To secure your
spot, please register using the link below:

[REGISTER_LINK]

We look forward to your participation.
Best regards,  
[Your Name]  
[Your Position]  
[Your Organization]


## 11. Pattern 4: Principles-Based Critique — *the Constitution*

The deck's bridge: a **constitution** of stable principles arching over a **critic agent**.

| Principle | Meaning |
|---|---|
| **Accuracy** | Do not invent facts |
| **Usefulness** | Answer directly and concretely |
| **Safety** | Avoid harmful/illegal guidance |
| **Format** | Respect required structure |


In [51]:
CONSTITUTION = {
    "accuracy":   "Do not invent facts. Every specific claim (dates, prices, statistics, names) "
                  "must come from the brief or be clearly marked as a placeholder.",
    "usefulness": "Answer directly and concretely. No filler, no restating the question.",
    "safety":     "Avoid harmful, illegal, discriminatory, or misleading guidance. "
                  "No manipulative urgency ('only 2 seats left!') unless stated in the brief.",
    "format":     "Respect the required structure and length constraints exactly.",
}



def constitutional_critique(draft: str, spec: TaskSpec, constitution: dict = CONSTITUTION,
                            threshold: int = 8) -> dict:
    """Pattern 4: evaluate against task rubric AND the reusable constitution."""
    const_text = "\n".join(f"  - {k.upper()}: {v}" for k, v in constitution.items())
    prompt = f"""{spec.render()}

THE CONSTITUTION (universal principles — violations are automatic failures):
{const_text}

DRAFT:
\"\"\"{draft}\"\"\"

Evaluate the draft against the brief AND this constitution.
Return JSON: {{"score": int 0-10, "passed": bool (false if ANY principle is violated),
"principle_violations": [{{"principle": str, "evidence": str}}],
"issues": [str], "revision_instructions": str}}"""
    return llm_json(prompt, system=CRITIC_SYSTEM)

In [52]:
# Bait the constitution: a draft that invents facts and manufactures urgency
violating_draft = """Subject: Amazing GenAI Workshop — 97% of our alumni got promoted!

Hi there,
Join 500+ professionals this Saturday. Industry legend Andrew Ng personally endorses
our curriculum. Only 2 seats left — register in the next 10 minutes or lose your spot!
Fee: just ₹499 (usually ₹49,999).
"""

verdict = constitutional_critique(violating_draft, EMAIL_SPEC)
print(f"score={verdict['score']}  passed={verdict['passed']}\n")
print("PRINCIPLE VIOLATIONS:")
for v in verdict["principle_violations"]:
    print(f"  ⚖️ {v['principle']}: {v['evidence']}")
print("\nrevision_instructions:")
show(verdict["revision_instructions"])

score=2  passed=False

PRINCIPLE VIOLATIONS:
  ⚖️ clarity: The date, format, and location are not clearly stated within the first five seconds.
  ⚖️ specificity: No concrete learning outcome is mentioned.
  ⚖️ factual_risk: The claim of '97% of our alumni got promoted' is an overpromise.
  ⚖️ audience_fit: The draft does not reference the specific work context of mid-career tech professionals.
  ⚖️ cta: The call to action is vague and lacks a clear link placeholder.

revision_instructions:
Rewrite to include the date (Sat 25 July 2026), format (in-person, Bengaluru, 10am-5pm), a specific
learning outcome, and a clear call to action with a link placeholder. Remove hype language and
ensure all claims are factual.


In [53]:
llm(f"Improve this violating_draft with the given instructions: {verdict['revision_instructions']}")

'**Draft: Workshop on Data Analysis Techniques**\n\nJoin us for an in-person workshop on Data Analysis Techniques on Saturday, 25 July 2026, from 10 AM to 5 PM in Bengaluru. This workshop is designed for individuals looking to enhance their skills in data analysis using practical tools and methodologies.\n\n**Learning Outcome:** By the end of this workshop, participants will be able to apply various data analysis techniques to real-world datasets, enabling them to draw actionable insights and make data-driven decisions.\n\nTo secure your spot, please register at [link_placeholder]. We look forward to seeing you there and helping you advance your data analysis skills.'